In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from PIL import Image
import timm

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True  # Otimização para A100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Caminhos
DRIVE_BASE = '/content/drive/MyDrive/dogs'
WORK_DIR = '/content/dogs'
DATASET_DIR = os.path.join(WORK_DIR, 'dog')

# Controle global do experimento
SPLIT_SAMPLE_PCT = 10.0   # Ex.: 50.0 => usa 50% de cada split
SPLIT_SAMPLE_FRAC = SPLIT_SAMPLE_PCT / 100.0
assert 0 < SPLIT_SAMPLE_FRAC <= 1.0, "SPLIT_SAMPLE_PCT deve estar em (0, 100]."

# Nomes dos arquivos de split (nova organização sem leakage)
SPLIT_FILES = {
    'train_identification': 'train_identification_60pct_ids.csv',
    'val_identification': 'val_identification.txt',
    'test_identification': 'test_identification.txt',
    'verification_val': 'verification_val_pairs.csv',
    'verification_test': 'verification_test_pairs.csv',
    'annotations': 'dog.csv',
}

# Criar diretório de trabalho
os.makedirs(WORK_DIR, exist_ok=True)

# Extrair dataset (se ainda não extraído)
if not os.path.exists(DATASET_DIR):
    print("Extraindo dataset...")
    archive_candidates = [
        os.path.join(DRIVE_BASE, 'dog.zip'),
        os.path.join(DRIVE_BASE, 'dog.tar.gz'),
        os.path.join(DRIVE_BASE, 'dog.tgz'),
    ]
    archive_path = next((p for p in archive_candidates if os.path.exists(p)), None)

    if archive_path is None:
        print("⚠️ Nenhum arquivo compactado do dataset foi encontrado no Drive.")
        print("   Certifique-se de que a pasta de imagens já está disponível em:", DATASET_DIR)
    else:
        if archive_path.endswith('.zip'):
            !unzip -q "{archive_path}" -d "{WORK_DIR}"
        else:
            !tar -xzf "{archive_path}" -C "{WORK_DIR}"

print(f"WORK_DIR: {WORK_DIR}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"SPLIT_SAMPLE_PCT: {SPLIT_SAMPLE_PCT:.1f}%")


In [ ]:
# Copiar arquivos de split do Drive
SPLIT_DIR = os.path.join(DRIVE_BASE, 'split')
ANNOTATIONS_CSV = os.path.join(DRIVE_BASE, SPLIT_FILES['annotations'])

# Se os splits estiverem no drive, copiar; caso contrário usar paths diretos
import shutil

LOCAL_SPLIT_DIR = os.path.join(WORK_DIR, 'split')
os.makedirs(LOCAL_SPLIT_DIR, exist_ok=True)

required_split_files = [
    SPLIT_FILES['train_identification'],
    SPLIT_FILES['val_identification'],
    SPLIT_FILES['test_identification'],
    SPLIT_FILES['verification_val'],
    SPLIT_FILES['verification_test'],
]

for fname in required_split_files:
    src = os.path.join(SPLIT_DIR, fname)
    dst = os.path.join(LOCAL_SPLIT_DIR, fname)
    if os.path.exists(src):
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            print(f"Copiado: {fname}")
    else:
        print(f"⚠️ Arquivo não encontrado no Drive: {src}")

if os.path.exists(ANNOTATIONS_CSV):
    local_ann = os.path.join(WORK_DIR, SPLIT_FILES['annotations'])
    if not os.path.exists(local_ann):
        shutil.copy2(ANNOTATIONS_CSV, local_ann)
        print("Copiado: dog.csv")
else:
    print(f"⚠️ dog.csv não encontrado em: {ANNOTATIONS_CSV}")


In [ ]:
# Carregar anotações
ann_path = os.path.join(WORK_DIR, SPLIT_FILES['annotations'])
df_ann = pd.read_csv(ann_path)
print(f"Total de indivíduos no CSV: {len(df_ann)}")
print(f"Colunas: {list(df_ann.columns)}")
display(df_ann.head(10))

def extract_identity_from_path(path):
    path = str(path).replace('\\', '/').strip()
    return os.path.basename(os.path.dirname(path))

def sample_identities(ids, frac, random_state=SEED):
    ids = list(dict.fromkeys(ids))
    if frac >= 1.0:
        return ids
    rng = random.Random(random_state)
    n = max(1, int(round(len(ids) * frac)))
    n = min(n, len(ids))
    idxs = sorted(rng.sample(range(len(ids)), n))
    return [ids[i] for i in idxs]

def sample_identification_dataframe_by_id(df, frac, label_col='label', random_state=SEED):
    if frac >= 1.0:
        return df.copy().reset_index(drop=True)
    selected_ids = sample_identities(df[label_col].drop_duplicates().tolist(), frac, random_state=random_state)
    out = df[df[label_col].isin(selected_ids)].copy().reset_index(drop=True)
    return out

def sample_identification_list_by_id(file_list, frac, random_state=SEED):
    if frac >= 1.0:
        return list(file_list)

    id_to_files = {}
    for fp in file_list:
        pet_id = extract_identity_from_path(fp)
        id_to_files.setdefault(pet_id, []).append(fp)

    selected_ids = set(sample_identities(list(id_to_files.keys()), frac, random_state=random_state))
    out = []
    for pet_id, files in id_to_files.items():
        if pet_id in selected_ids:
            out.extend(files)
    return out

def add_pair_id_columns(df):
    out = df.copy()
    out['id1'] = out['filename1'].map(extract_identity_from_path)
    out['id2'] = out['filename2'].map(extract_identity_from_path)
    return out

def sample_verification_pairs_by_identity(df, frac, random_state=SEED):
    df = add_pair_id_columns(df)
    unique_ids = sorted(set(df['id1']).union(set(df['id2'])))

    if frac < 1.0:
        selected_ids = set(sample_identities(unique_ids, frac, random_state=random_state))
        df = df[df['id1'].isin(selected_ids) & df['id2'].isin(selected_ids)].copy()
    else:
        selected_ids = set(unique_ids)

    if 'pair_type' in df.columns:
        summary_cols = ['label', 'pair_type']
    else:
        summary_cols = ['label']

    df = df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    print(f"IDs selecionados para verificação: {len(selected_ids)} / {len(unique_ids)}")
    print(df.groupby(summary_cols, dropna=False).size().to_string())
    return df

# Carregar splits com subamostragem opcional
train_path = os.path.join(LOCAL_SPLIT_DIR, SPLIT_FILES['train_identification'])
verif_val_path = os.path.join(LOCAL_SPLIT_DIR, SPLIT_FILES['verification_val'])
verif_test_path = os.path.join(LOCAL_SPLIT_DIR, SPLIT_FILES['verification_test'])

df_train = pd.read_csv(train_path)
df_verif_val = pd.read_csv(verif_val_path)
df_verif_test = pd.read_csv(verif_test_path)

with open(os.path.join(LOCAL_SPLIT_DIR, SPLIT_FILES['val_identification']), 'r') as f:
    val_files = [l.strip() for l in f.readlines() if l.strip()]

with open(os.path.join(LOCAL_SPLIT_DIR, SPLIT_FILES['test_identification']), 'r') as f:
    test_files = [l.strip() for l in f.readlines() if l.strip()]

# Aplicar amostragem global em todos os splits
# Identificação: amostrar identidades inteiras, preservando todas as imagens de cada pet.
df_train = sample_identification_dataframe_by_id(df_train, SPLIT_SAMPLE_FRAC, label_col='label')
val_files = sample_identification_list_by_id(val_files, SPLIT_SAMPLE_FRAC)
test_files = sample_identification_list_by_id(test_files, SPLIT_SAMPLE_FRAC)

# Verificação: amostrar identidades inteiras do split e manter apenas pares compostos
# exclusivamente por IDs selecionados, evitando quebrar pares positivos.
df_verif_val = sample_verification_pairs_by_identity(df_verif_val, SPLIT_SAMPLE_FRAC)
df_verif_test = sample_verification_pairs_by_identity(df_verif_test, SPLIT_SAMPLE_FRAC)

print(f"Train identification: {len(df_train)} imagens, {df_train['label'].nunique()} identidades")
print(f"Val identification:   {len(val_files)} imagens, {len({extract_identity_from_path(x) for x in val_files})} identidades")
print(f"Test identification:  {len(test_files)} imagens, {len({extract_identity_from_path(x) for x in test_files})} identidades")
print(f"Verification val:     {len(df_verif_val)} pares, {df_verif_val['id1'].nunique()} identidades")
print(f"Verification test:    {len(df_verif_test)} pares, {df_verif_test['id1'].nunique()} identidades")


In [ ]:
# Distribuição de imagens por identidade (treino)
imgs_per_id_train = df_train.groupby('label').size()
print(f"\nImagens por identidade (treino):")
print(f"  Mín: {imgs_per_id_train.min()}, Máx: {imgs_per_id_train.max()}, Média: {imgs_per_id_train.mean():.1f}, Mediana: {imgs_per_id_train.median():.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(imgs_per_id_train.values, bins=30, color='#4C72B0', edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Imagens por identidade')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Imagens por Identidade (Train)')

# Distribuição de raças
breed_counts = df_ann['Breed'].value_counts().head(20)
axes[1].barh(breed_counts.index[::-1], breed_counts.values[::-1], color='#55A868', edgecolor='white')
axes[1].set_xlabel('Contagem')
axes[1].set_title('Top 20 Raças')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'eda_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Verificação: estatísticas dos splits de validation/test
def summarize_verification_df(name, df):
    pos_count = int((df['label'] == 1).sum())
    neg_count = int((df['label'] == 0).sum())
    print(f"\n{name}:")
    print(f"  Positivos: {pos_count} ({100*pos_count/len(df):.1f}%)")
    print(f"  Negativos: {neg_count} ({100*neg_count/len(df):.1f}%)")
    if 'pair_type' in df.columns:
        print(df['pair_type'].value_counts(dropna=False).to_string())

summarize_verification_df('Verification Val', df_verif_val)
summarize_verification_df('Verification Test', df_verif_test)

# Visualizar exemplos de pares do verification_val
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Exemplos de Pares de Verificação (validation)', fontsize=16, fontweight='bold')

pos_pairs = df_verif_val[df_verif_val['label'] == 1].sample(3, random_state=SEED)
for i, (_, row) in enumerate(pos_pairs.iterrows()):
    img1 = Image.open(os.path.join(WORK_DIR, row['filename1']))
    img2 = Image.open(os.path.join(WORK_DIR, row['filename2']))
    axes[0, i*2].imshow(img1)
    axes[0, i*2].set_title('Mesmo cão ✓', color='green', fontweight='bold')
    axes[0, i*2].axis('off')
    axes[0, i*2+1].imshow(img2)
    axes[0, i*2+1].set_title('Mesmo cão ✓', color='green', fontweight='bold')
    axes[0, i*2+1].axis('off')

neg_pairs = df_verif_val[df_verif_val['label'] == 0].sample(3, random_state=SEED)
for i, (_, row) in enumerate(neg_pairs.iterrows()):
    img1 = Image.open(os.path.join(WORK_DIR, row['filename1']))
    img2 = Image.open(os.path.join(WORK_DIR, row['filename2']))
    axes[1, i*2].imshow(img1)
    axes[1, i*2].set_title('Cão diferente ✗', color='red', fontweight='bold')
    axes[1, i*2].axis('off')
    axes[1, i*2+1].imshow(img2)
    axes[1, i*2+1].set_title(row.get('pair_type', 'negativo'), color='red', fontweight='bold')
    axes[1, i*2+1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'verification_pairs_examples.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Transforms
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.2),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class PetFaceTrainDataset(Dataset):
    """Dataset para treino — carrega imagens individuais com label de identidade."""

    def __init__(self, data, root_dir, transform=None):
        if isinstance(data, (str, Path)):
            self.df = pd.read_csv(data)
        else:
            self.df = data.copy().reset_index(drop=True)

        self.root_dir = root_dir
        self.transform = transform

        # Remapear labels para 0..N-1 (contíguos)
        unique_labels = sorted(self.df['label'].unique())
        self.label_map = {old: new for new, old in enumerate(unique_labels)}
        self.num_classes = len(unique_labels)

        print(f"Train dataset: {len(self.df)} imagens, {self.num_classes} classes")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['filename'])
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = self.label_map[row['label']]
        return img, label


In [ ]:
class PetFaceVerificationDataset(Dataset):
    """Dataset para verificação — carrega pares de imagens."""

    def __init__(self, data, root_dir, transform=None):
        if isinstance(data, (str, Path)):
            self.df = pd.read_csv(data)
        else:
            self.df = data.copy().reset_index(drop=True)

        self.root_dir = root_dir
        self.transform = transform
        print(f"Verification dataset: {len(self.df)} pares")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img1 = Image.open(os.path.join(self.root_dir, row['filename1'])).convert('RGB')
        img2 = Image.open(os.path.join(self.root_dir, row['filename2'])).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        label = int(row['label'])
        return img1, img2, label


In [ ]:
# Criar datasets
train_dataset = PetFaceTrainDataset(
    data=df_train,
    root_dir=WORK_DIR,
    transform=train_transform
)

verif_val_dataset = PetFaceVerificationDataset(
    data=df_verif_val,
    root_dir=WORK_DIR,
    transform=eval_transform
)

verif_test_dataset = PetFaceVerificationDataset(
    data=df_verif_test,
    root_dir=WORK_DIR,
    transform=eval_transform
)

NUM_CLASSES = train_dataset.num_classes
print(f"\nNúmero de identidades (classes): {NUM_CLASSES}")

# Observação importante:
print("\nNota: 'val_identification' e 'test_identification' têm IDs disjuntos do treino.")
print("Por isso, eles NÃO medem accuracy fechada da cabeça de classificação.")
print("A seleção de checkpoint e do threshold será feita em verification_val.")

print("A subamostragem percentual é feita por identidade, não por imagem.")

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 4

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

if NUM_WORKERS > 0:
    loader_kwargs.update(dict(prefetch_factor=2, persistent_workers=True))

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

verif_val_loader = DataLoader(
    verif_val_dataset,
    shuffle=False,
    **loader_kwargs,
)

verif_test_loader = DataLoader(
    verif_test_dataset,
    shuffle=False,
    **loader_kwargs,
)

print(f"Train: {len(train_loader)} batches de {BATCH_SIZE}")
print(f"Verification val: {len(verif_val_loader)} batches de {BATCH_SIZE}")
print(f"Verification test: {len(verif_test_loader)} batches de {BATCH_SIZE}")


In [ ]:
class EmbeddingModel(nn.Module):
    """ConvNeXt Small backbone + embedding head."""

    def __init__(self, embedding_dim=512, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_small.fb_in22k_ft_in1k',
            pretrained=pretrained,
            num_classes=0,  # Remove classifier, retorna features
        )
        backbone_dim = self.backbone.num_features
        print(f"Backbone output dim: {backbone_dim}")

        self.embedding = nn.Sequential(
            nn.Linear(backbone_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features = self.backbone(x)
        embeddings = self.embedding(features)
        # L2 normalize
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings


class ArcFaceHead(nn.Module):
    """ArcFace (Additive Angular Margin) classification head."""

    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s = s      # scale
        self.m = m      # margin (em radianos)
        self.num_classes = num_classes

        # Pesos da camada (representam os centros das classes no espaço de embeddings)
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, embeddings, labels):
        # Normalizar pesos
        W = F.normalize(self.weight, p=2, dim=1)
        # Cosine similarity
        cosine = F.linear(embeddings, W)
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)

        # Calcular sin(theta)
        sine = torch.sqrt(1.0 - cosine.pow(2))

        # cos(theta + m) = cos(theta)*cos(m) - sin(theta)*sin(m)
        phi = cosine * self.cos_m - sine * self.sin_m

        # Se cos(theta) < cos(pi - m), usar fallback para estabilidade
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # One-hot dos labels
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        # Aplicar margem apenas na classe correta
        output = one_hot * phi + (1.0 - one_hot) * cosine
        output = output * self.s

        return output


class SoftmaxModel(nn.Module):
    """Modelo com Softmax loss — classificação direta por identidade."""

    def __init__(self, embedding_dim, num_classes):
        super().__init__()
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, embeddings):
        return self.classifier(embeddings)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

def compute_metrics_from_scores(labels, sims, threshold=None):
    labels = np.asarray(labels).astype(int)
    sims = np.asarray(sims).astype(np.float32)

    auc = roc_auc_score(labels, sims)
    fpr, tpr, thresholds = roc_curve(labels, sims)

    if threshold is None:
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        threshold = float(thresholds[best_idx])

    preds = (sims >= threshold).astype(int)
    acc = accuracy_score(labels, preds)

    return {
        'auc': float(auc),
        'accuracy': float(acc),
        'threshold': float(threshold),
        'fpr': fpr,
        'tpr': tpr,
        'similarities': sims,
        'labels': labels,
    }

def add_intra_breed_accuracy(metrics, df_pairs, threshold):
    out = dict(metrics)
    out['intra_breed_accuracy'] = np.nan
    out['intra_breed_count'] = 0

    if 'pair_type' not in df_pairs.columns:
        return out

    intra_mask = df_pairs['pair_type'].eq('negative_same_breed').to_numpy()
    intra_count = int(intra_mask.sum())
    out['intra_breed_count'] = intra_count

    if intra_count > 0:
        sims = np.asarray(metrics['similarities'])[intra_mask]
        labels = np.asarray(metrics['labels'])[intra_mask]
        preds = (sims >= threshold).astype(int)
        out['intra_breed_accuracy'] = float(accuracy_score(labels, preds))

    return out

def infer_pair_stats(model, verif_loader, device):
    model.eval()
    all_sims = []
    all_labels = []

    with torch.no_grad():
        for img1, img2, labels in tqdm(verif_loader, desc='Avaliando', leave=False):
            img1, img2 = img1.to(device), img2.to(device)

            with autocast(dtype=torch.float16):
                emb1 = model(img1)
                emb2 = model(img2)

            similarity = F.cosine_similarity(emb1, emb2, dim=1)
            all_sims.extend(similarity.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_sims), np.array(all_labels)

def evaluate_verification(model, verif_loader, device, df_pairs=None, threshold=None):
    """Avalia o modelo em verificação usando cosine similarity.

    - Se threshold=None, estima o melhor limiar no próprio split (uso típico: verification_val).
    - Se threshold é fornecido, usa esse limiar fixo (uso típico: verification_test).
    """
    all_sims, all_labels = infer_pair_stats(model, verif_loader, device)
    metrics = compute_metrics_from_scores(all_labels, all_sims, threshold=threshold)

    if df_pairs is not None:
        metrics = add_intra_breed_accuracy(metrics, df_pairs.reset_index(drop=True), metrics['threshold'])

    return metrics

def train_one_epoch(model, head, train_loader, optimizer, scaler, device, loss_type='softmax'):
    """Treina uma época."""
    model.train()
    head.train()
    total_loss = 0
    correct = 0
    total = 0
    criterion = nn.CrossEntropyLoss()

    for imgs, labels in tqdm(train_loader, desc='Treinando', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(dtype=torch.float16):
            embeddings = model(imgs)

            if loss_type == 'arcface':
                logits = head(embeddings, labels)
            else:
                logits = head(embeddings)

            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(head.parameters()), max_norm=5.0
        )
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    avg_loss = total_loss / total
    avg_acc = correct / total
    return avg_loss, avg_acc


In [ ]:
print("=" * 60)
print("EXPERIMENTO 1: SOFTMAX (CrossEntropy)")
print("=" * 60)

# Modelo
softmax_backbone = EmbeddingModel(embedding_dim=512, pretrained=True).to(DEVICE)
softmax_head = SoftmaxModel(embedding_dim=512, num_classes=NUM_CLASSES).to(DEVICE)

# Otimizador
optimizer_softmax = torch.optim.AdamW(
    list(softmax_backbone.parameters()) + list(softmax_head.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

NUM_EPOCHS = 20
scheduler_softmax = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_softmax, T_max=NUM_EPOCHS, eta_min=1e-6
)
scaler_softmax = GradScaler()

# Treino
softmax_history = {'loss': [], 'acc': [], 'verif_val_auc': [], 'verif_val_acc': [], 'lr': []}
best_auc_softmax = -np.inf

for epoch in range(1, NUM_EPOCHS + 1):
    loss, acc = train_one_epoch(
        softmax_backbone, softmax_head, train_loader,
        optimizer_softmax, scaler_softmax, DEVICE, loss_type='softmax'
    )

    # verification_val é usado para seleção de checkpoint e threshold
    val_results = evaluate_verification(
        softmax_backbone, verif_val_loader, DEVICE, df_pairs=df_verif_val, threshold=None
    )

    lr = optimizer_softmax.param_groups[0]['lr']
    softmax_history['loss'].append(loss)
    softmax_history['acc'].append(acc)
    softmax_history['verif_val_auc'].append(val_results['auc'])
    softmax_history['verif_val_acc'].append(val_results['accuracy'])
    softmax_history['lr'].append(lr)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {loss:.4f} | "
        f"Train Acc: {acc:.4f} | VerifVal AUC: {val_results['auc']:.4f} | "
        f"VerifVal Acc@best_t: {val_results['accuracy']:.4f} | "
        f"Intra-breed Acc: {val_results['intra_breed_accuracy']:.4f} | "
        f"LR: {lr:.2e}"
    )

    if val_results['auc'] > best_auc_softmax:
        best_auc_softmax = val_results['auc']
        torch.save({
            'backbone': softmax_backbone.state_dict(),
            'head': softmax_head.state_dict(),
            'epoch': epoch,
            'verif_val_auc': val_results['auc'],
            'verif_val_threshold': val_results['threshold'],
            'split_sample_pct': SPLIT_SAMPLE_PCT,
        }, os.path.join(DRIVE_BASE, 'best_softmax.pth'))
        print(f"  → Melhor modelo salvo! VerifVal AUC: {val_results['auc']:.4f}")

    scheduler_softmax.step()

print(f"\nMelhor AUC em verification_val (Softmax): {best_auc_softmax:.4f}")


In [ ]:
# Carregar melhor modelo Softmax para avaliação final em verification_test
ckpt = torch.load(os.path.join(DRIVE_BASE, 'best_softmax.pth'), map_location=DEVICE, weights_only=False)
softmax_backbone.load_state_dict(ckpt['backbone'])
softmax_threshold = float(ckpt['verif_val_threshold'])

softmax_val_results = evaluate_verification(
    softmax_backbone, verif_val_loader, DEVICE, df_pairs=df_verif_val, threshold=softmax_threshold
)
softmax_results = evaluate_verification(
    softmax_backbone, verif_test_loader, DEVICE, df_pairs=df_verif_test, threshold=softmax_threshold
)

print(
    f"Softmax — TEST AUC: {softmax_results['auc']:.4f} | "
    f"Test Acc@val_threshold: {softmax_results['accuracy']:.4f} | "
    f"Intra-breed Acc: {softmax_results['intra_breed_accuracy']:.4f} | "
    f"Threshold (from val): {softmax_threshold:.4f}"
)


In [ ]:
print("=" * 60)
print("EXPERIMENTO 2: ARCFACE (Additive Angular Margin)")
print("=" * 60)
NUM_EPOCHS = 20

arcface_backbone = EmbeddingModel(embedding_dim=512, pretrained=True).to(DEVICE)
arcface_head = ArcFaceHead(
    embedding_dim=512,
    num_classes=NUM_CLASSES,
    s=30.0,
    m=0.50,
).to(DEVICE)

optimizer_arcface = torch.optim.AdamW(
    list(arcface_backbone.parameters()) + list(arcface_head.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler_arcface = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_arcface, T_max=NUM_EPOCHS, eta_min=1e-6
)
scaler_arcface = GradScaler()

arcface_history = {'loss': [], 'acc': [], 'verif_val_auc': [], 'verif_val_acc': [], 'lr': []}
best_auc_arcface = -np.inf

for epoch in range(1, NUM_EPOCHS + 1):
    loss, acc = train_one_epoch(
        arcface_backbone, arcface_head, train_loader,
        optimizer_arcface, scaler_arcface, DEVICE, loss_type='arcface'
    )

    val_results = evaluate_verification(
        arcface_backbone, verif_val_loader, DEVICE, df_pairs=df_verif_val, threshold=None
    )

    lr = optimizer_arcface.param_groups[0]['lr']
    arcface_history['loss'].append(loss)
    arcface_history['acc'].append(acc)
    arcface_history['verif_val_auc'].append(val_results['auc'])
    arcface_history['verif_val_acc'].append(val_results['accuracy'])
    arcface_history['lr'].append(lr)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | Loss: {loss:.4f} | "
        f"Train Acc: {acc:.4f} | VerifVal AUC: {val_results['auc']:.4f} | "
        f"VerifVal Acc@best_t: {val_results['accuracy']:.4f} | "
        f"Intra-breed Acc: {val_results['intra_breed_accuracy']:.4f} | "
        f"LR: {lr:.2e}"
    )

    if val_results['auc'] > best_auc_arcface:
        best_auc_arcface = val_results['auc']
        torch.save({
            'backbone': arcface_backbone.state_dict(),
            'head': arcface_head.state_dict(),
            'epoch': epoch,
            'verif_val_auc': val_results['auc'],
            'verif_val_threshold': val_results['threshold'],
            'split_sample_pct': SPLIT_SAMPLE_PCT,
        }, os.path.join(DRIVE_BASE, 'best_arcface.pth'))
        print(f"  → Melhor modelo salvo! VerifVal AUC: {val_results['auc']:.4f}")

    scheduler_arcface.step()

print(f"\nMelhor AUC em verification_val (ArcFace): {best_auc_arcface:.4f}")


In [ ]:
ckpt = torch.load(os.path.join(DRIVE_BASE, 'best_arcface.pth'), map_location=DEVICE, weights_only=False)
arcface_backbone.load_state_dict(ckpt['backbone'])
arcface_threshold = float(ckpt['verif_val_threshold'])


In [ ]:
arcface_val_results = evaluate_verification(
    arcface_backbone, verif_val_loader, DEVICE, df_pairs=df_verif_val, threshold=arcface_threshold
)
arcface_results = evaluate_verification(
    arcface_backbone, verif_test_loader, DEVICE, df_pairs=df_verif_test, threshold=arcface_threshold
)

print(
    f"ArcFace — TEST AUC: {arcface_results['auc']:.4f} | "
    f"Test Acc@val_threshold: {arcface_results['accuracy']:.4f} | "
    f"Intra-breed Acc: {arcface_results['intra_breed_accuracy']:.4f} | "
    f"Threshold (from val): {arcface_threshold:.4f}"
)


In [ ]:
# Tabela comparativa
print("\n" + "=" * 90)
print("RESULTADOS FINAIS — VERIFICAÇÃO DE CÃES (TESTE FINAL)")
print("=" * 90)

results_data = {
    'Método': [
        'Softmax (ConvNeXt-S)',
        'ArcFace (ConvNeXt-S)',
    ],
    'Val AUC (%)': [
        f"{softmax_val_results['auc']*100:.2f}",
        f"{arcface_val_results['auc']*100:.2f}",
    ],
    'Test AUC (%)': [
        f"{softmax_results['auc']*100:.2f}",
        f"{arcface_results['auc']*100:.2f}",
    ],
    'Test Acc (%)': [
        f"{softmax_results['accuracy']*100:.2f}",
        f"{arcface_results['accuracy']*100:.2f}",
    ],
    'Intra-breed Acc (%)': [
        f"{softmax_results['intra_breed_accuracy']*100:.2f}",
        f"{arcface_results['intra_breed_accuracy']*100:.2f}",
    ],
    'Threshold (val)': [
        f"{softmax_threshold:.4f}",
        f"{arcface_threshold:.4f}",
    ],
}

df_results = pd.DataFrame(results_data)
print(df_results.to_string(index=False))


In [ ]:
# Curvas ROC no conjunto final de verification_test
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].plot(
    softmax_results['fpr'], softmax_results['tpr'],
    label=f"Softmax (AUC={softmax_results['auc']:.4f})", linewidth=2, color='#4C72B0'
)
axes[0].plot(
    arcface_results['fpr'], arcface_results['tpr'],
    label=f"ArcFace (AUC={arcface_results['auc']:.4f})", linewidth=2, color='#DD8452'
)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('Curva ROC — Verification Test', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11, loc='lower right')
axes[0].grid(True, alpha=0.3)

pos_sims_s = softmax_results['similarities'][softmax_results['labels'] == 1]
neg_sims_s = softmax_results['similarities'][softmax_results['labels'] == 0]
axes[1].hist(neg_sims_s, bins=60, alpha=0.7, label='Negativo', color='#55A868', density=True)
axes[1].hist(pos_sims_s, bins=60, alpha=0.7, label='Positivo', color='#ECBF6A', density=True)
axes[1].axvline(softmax_threshold, color='red', linestyle='--', alpha=0.8,
                label=f"Threshold={softmax_threshold:.3f}")
axes[1].set_xlabel('Similaridade Coseno', fontsize=12)
axes[1].set_ylabel('Densidade', fontsize=12)
axes[1].set_title('Softmax — Similaridades (test)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

pos_sims_a = arcface_results['similarities'][arcface_results['labels'] == 1]
neg_sims_a = arcface_results['similarities'][arcface_results['labels'] == 0]
axes[2].hist(neg_sims_a, bins=60, alpha=0.7, label='Negativo', color='#55A868', density=True)
axes[2].hist(pos_sims_a, bins=60, alpha=0.7, label='Positivo', color='#ECBF6A', density=True)
axes[2].axvline(arcface_threshold, color='red', linestyle='--', alpha=0.8,
                label=f"Threshold={arcface_threshold:.3f}")
axes[2].set_xlabel('Similaridade Coseno', fontsize=12)
axes[2].set_ylabel('Densidade', fontsize=12)
axes[2].set_title('ArcFace — Similaridades (test)', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, 'results_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Curvas de treino (seleção por verification_val)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(softmax_history['loss'], label='Softmax', linewidth=2, color='#4C72B0')
axes[0].plot(arcface_history['loss'], label='ArcFace', linewidth=2, color='#DD8452')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(softmax_history['acc'], label='Softmax', linewidth=2, color='#4C72B0')
axes[1].plot(arcface_history['acc'], label='ArcFace', linewidth=2, color='#DD8452')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(softmax_history['verif_val_auc'], label='Softmax', linewidth=2, color='#4C72B0')
axes[2].plot(arcface_history['verif_val_auc'], label='ArcFace', linewidth=2, color='#DD8452')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('AUC')
axes[2].set_title('Verification Val AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def visualize_predictions(model, df, threshold, device, num_pairs=10):
    model.eval()

    num_pos = num_pairs // 2
    num_neg = num_pairs - num_pos
    pos_pairs = df[df['label'] == 1].sample(min(num_pos, (df['label'] == 1).sum()), random_state=SEED)
    neg_pairs = df[df['label'] == 0].sample(min(num_neg, (df['label'] == 0).sum()), random_state=SEED)
    sample_pairs = pd.concat([pos_pairs, neg_pairs]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(8, 3.5 * len(sample_pairs)))
    if len(sample_pairs) == 1:
        axes = np.array([axes])
    fig.suptitle('Exemplos de Predições de Verificação (ArcFace — Teste)', fontsize=16, fontweight='bold', y=0.98)

    with torch.no_grad():
        for i, (_, row) in enumerate(sample_pairs.iterrows()):
            img1_path = os.path.join(WORK_DIR, row['filename1'])
            img2_path = os.path.join(WORK_DIR, row['filename2'])

            img1_pil = Image.open(img1_path).convert('RGB')
            img2_pil = Image.open(img2_path).convert('RGB')

            img1_t = eval_transform(img1_pil).unsqueeze(0).to(device)
            img2_t = eval_transform(img2_pil).unsqueeze(0).to(device)

            with autocast(dtype=torch.float16):
                emb1 = model(img1_t)
                emb2 = model(img2_t)

            sim = F.cosine_similarity(emb1, emb2).item()
            pred_match = sim >= threshold
            true_match = row['label'] == 1

            color = 'green' if pred_match == true_match else 'red'
            status = 'RESULTADO: ACERTOU' if pred_match == true_match else 'RESULTADO: ERROU'
            pair_desc = row.get('pair_type', 'n/a')

            axes[i, 0].imshow(img1_pil)
            axes[i, 0].axis('off')
            axes[i, 0].set_title(f"Label Real: {'Mesmo Cão' if true_match else 'Cães Diferentes'}", fontsize=11)

            axes[i, 1].imshow(img2_pil)
            axes[i, 1].axis('off')
            axes[i, 1].set_title(
                f"Predição: {'Mesmo Cão' if pred_match else 'Cães Diferentes'}\n"
                f"Tipo: {pair_desc} | Similaridade: {sim:.3f} | Threshold: {threshold:.3f}\n"
                f"{status}",
                color=color, fontsize=11, fontweight='bold'
            )

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = os.path.join(WORK_DIR, 'prediction_examples_results.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

print(f"Visualizando predições com ArcFace no verification_test (Threshold do val: {arcface_threshold:.3f})\n")
visualize_predictions(arcface_backbone, df_verif_test, arcface_threshold, DEVICE, num_pairs=20)


In [ ]:
# Salvar resultados finais em CSV
df_final = pd.DataFrame({
    'Método': ['Softmax (ConvNeXt-S)', 'ArcFace (ConvNeXt-S)'],
    'Split sample pct': [SPLIT_SAMPLE_PCT, SPLIT_SAMPLE_PCT],
    'Val AUC': [softmax_val_results['auc'], arcface_val_results['auc']],
    'Test AUC': [softmax_results['auc'], arcface_results['auc']],
    'Test Accuracy': [softmax_results['accuracy'], arcface_results['accuracy']],
    'Intra-breed Accuracy': [softmax_results['intra_breed_accuracy'], arcface_results['intra_breed_accuracy']],
    'Threshold (from val)': [softmax_threshold, arcface_threshold],
    'Intra-breed Pairs (test)': [softmax_results['intra_breed_count'], arcface_results['intra_breed_count']],
})
df_final.to_csv(os.path.join(WORK_DIR, 'verification_results.csv'), index=False)

drive_results_path = os.path.join(DRIVE_BASE, 'verification_results.csv')
df_final.to_csv(drive_results_path, index=False)
print(f"Resultados salvos em: {drive_results_path}")

for fname in ['results_comparison.png', 'training_curves.png',
              'eda_distributions.png', 'verification_pairs_examples.png', 'prediction_examples_results.png']:
    src = os.path.join(WORK_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_BASE, fname))

print("Gráficos salvos no Drive!")
